# 01 - Data Understanding

Telco customer churn data for a California telecom provider. Each row is one customer.
Goal of this notebook: understand structure, quality and the target before touching anything.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

RAW_PATH = "../Data/Raw/Telco_customer_churn.xlsx"
df = pd.read_excel(RAW_PATH)
print(df.shape)

(7043, 33)


## Structure

In [2]:
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [3]:
df.dtypes

CustomerID               str
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str

In [4]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
})
summary

,dtype,n_unique,n_missing
CustomerID,str,7043,0
Count,int64,1,0
Country,str,1,0
State,str,1,0
City,str,1129,0
Zip Code,int64,1652,0
Lat Long,str,1652,0
Latitude,float64,1652,0
Longitude,float64,1651,0
Gender,str,2,0


## Column groups

- **Identifiers / location:** CustomerID, Count, Country, State, City, Zip Code, Lat Long, Latitude, Longitude
- **Demographics:** Gender, Senior Citizen, Partner, Dependents
- **Account:** Tenure Months, Contract, Paperless Billing, Payment Method, Monthly Charges, Total Charges
- **Services:** Phone Service, Multiple Lines, Internet Service, Online Security, Online Backup, Device Protection, Tech Support, Streaming TV, Streaming Movies
- **Target / churn outcome:** Churn Label, Churn Value, Churn Score, CLTV, Churn Reason

## Target variable

In [5]:
counts = df["Churn Label"].value_counts()
rate = df["Churn Value"].mean()
print(counts)
print(f"\nChurn rate: {rate:.2%}")

Churn Label
No     5174
Yes    1869
Name: count, dtype: int64

Churn rate: 26.54%


The classes are imbalanced (roughly 27% churn), so accuracy alone will be misleading later. Recall, precision and ROC-AUC matter more.

## Data quality checks

In [6]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate CustomerIDs:", df["CustomerID"].duplicated().sum())
print("Constant columns:", [c for c in df.columns if df[c].nunique() == 1])

Duplicate rows: 0
Duplicate CustomerIDs: 0
Constant columns: ['Count', 'Country', 'State']


In [7]:
# Total Charges is stored as text - find the values that don't convert
tc = pd.to_numeric(df["Total Charges"], errors="coerce")
bad = df[tc.isna()]
print(len(bad), "rows with non-numeric Total Charges")
bad[["CustomerID", "Tenure Months", "Monthly Charges", "Total Charges", "Churn Label"]]

11 rows with non-numeric Total Charges


,CustomerID,Tenure Months,Monthly Charges,Total Charges,Churn Label
2234,4472-LVYGI,0,52.55,,No
2438,3115-CZMZD,0,20.25,,No
2568,5709-LVOEQ,0,80.85,,No
2667,4367-NUYAO,0,25.75,,No
2856,1371-DWPAZ,0,56.05,,No
4331,7644-OMVMY,0,19.85,,No
4687,3213-VVOLG,0,25.35,,No
5104,2520-SGTTA,0,20.00,,No
5719,2923-ARZLG,0,19.70,,No
6772,4075-WKNIU,0,73.35,,No


All 11 bad rows are customers with `Tenure Months = 0` (just signed up, nothing billed yet) and none of them churned. Safe to treat as 0.

In [8]:
# Churn Reason only exists for churned customers
print("Churn Reason missing:", df["Churn Reason"].isna().sum())
print("Churned customers:", df["Churn Value"].sum())
print("Missing reasons == retained customers:", df["Churn Reason"].isna().sum() == (df["Churn Value"] == 0).sum())

Churn Reason missing: 5174
Churned customers: 1869
Missing reasons == retained customers: True


## Numeric overview

In [9]:
df[["Tenure Months", "Monthly Charges", "Churn Score", "CLTV"]].describe().round(2)

,Tenure Months,Monthly Charges,Churn Score,CLTV
count,7043.00,7043.00,7043.00,7043.00
mean,32.37,64.76,58.70,4400.30
std,24.56,30.09,21.53,1183.06
min,0.00,18.25,5.00,2003.00
25%,9.00,35.50,40.00,3469.00
50%,29.00,70.35,61.00,4527.00
75%,55.00,89.85,75.00,5380.50
max,72.00,118.75,100.00,6500.00


## Leakage check

`Churn Score` and `Churn Reason` are produced after (or alongside) the churn outcome, so they cannot be used as model inputs.

In [10]:
print(df.groupby("Churn Label")["Churn Score"].agg(["mean", "min", "max"]).round(1))
print()
print("Lowest score among churners :", df.loc[df["Churn Value"] == 1, "Churn Score"].min())
print("Highest score among retained:", df.loc[df["Churn Value"] == 0, "Churn Score"].max())

             mean  min  max
Churn Label                
No           50.1    5   80
Yes          82.5   65  100

Lowest score among churners : 65
Highest score among retained: 80


## Findings

- 7,043 customers, 33 columns, no duplicate customers.
- `Country`, `State` and `Count` are constant and add nothing.
- `Total Charges` needs a type fix (11 blanks, all tenure 0).
- Target is imbalanced: about 26.5% churned.
- `Churn Score`, `Churn Reason` (and the label columns) must be excluded from model features. `CLTV` is a modelled value, so it is kept for business analysis only.